In [8]:
import os

os.environ['HTTP_PROXY'] = '127.0.0.1:7890'
os.environ['HTTPS_PROXY'] = '127.0.0.1:7890'
import json

from urllib.request import urlopen
from PIL import Image
import torch
from huggingface_hub import hf_hub_download
from open_clip import create_model_and_transforms, get_tokenizer
from open_clip.factory import HF_HUB_PREFIX, _MODEL_CONFIGS


# Download the model and config files
hf_hub_download(
    repo_id="microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224",
    filename="open_clip_pytorch_model.bin",
    local_dir="checkpoints"
)
hf_hub_download(
    repo_id="microsoft/BiomedCLIP-PubMedBERT_256-vit_base_patch16_224",
    filename="open_clip_config.json",
    local_dir="checkpoints"
)


# Load the model and config files
model_name = "biomedclip_local"

with open("checkpoints/open_clip_config.json", "r") as f:
    config = json.load(f)
    model_cfg = config["model_cfg"]
    preprocess_cfg = config["preprocess_cfg"]


if (not model_name.startswith(HF_HUB_PREFIX)
    and model_name not in _MODEL_CONFIGS
    and config is not None):
    _MODEL_CONFIGS[model_name] = model_cfg

tokenizer = get_tokenizer(model_name)

model, _, preprocess = create_model_and_transforms(
    model_name=model_name,
    pretrained="checkpoints/open_clip_pytorch_model.bin",
    **{f"image_{k}": v for k, v in preprocess_cfg.items()},
)


In [9]:
from torchvision import transforms
import random
from collections import defaultdict
from torchvision.datasets import ImageFolder
from torch.utils.data import Dataset
from PIL import Image
import torch

class FewShotImageFolder(Dataset):
    def __init__(self, root, num_shots, transform=None, test_transform=None, seed=42):
        super().__init__()
        self.root = root
        self.num_shots = num_shots
        self.transform = transform
        self.test_transform = test_transform
        
        full_dataset = ImageFolder(root)
        self.class_to_idx = full_dataset.class_to_idx

        label_to_samples = defaultdict(list)
        for path, label in full_dataset.samples:
            label_to_samples[label].append(path)
        
        random.seed(seed)
        
        self.train_samples = []
        self.train_targets = []
        self.test_samples = []
        self.test_targets = []

        for label, samples in label_to_samples.items():
            if len(samples) <= num_shots:
                sampled = samples
                remain = []
            else:
                sampled = random.sample(samples, num_shots)
                remain = [s for s in samples if s not in sampled]

            self.train_samples.extend(sampled)
            self.train_targets.extend([label] * len(sampled))
            self.test_samples.extend(remain)
            self.test_targets.extend([label] * len(remain))

        # 默认使用训练集
        combined = list(zip(self.train_samples, self.train_targets))
        random.shuffle(combined)
        self.samples, self.targets = zip(*combined)
        self.samples = list(self.samples)
        self.targets = list(self.targets)
        
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        path = self.samples[idx]
        label = self.targets[idx]
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        label = torch.tensor(label)
        return image, label

    def get_test_set(self):
        return SimpleImageDataset(self.test_samples, self.test_targets,self.test_transform)


class SimpleImageDataset(Dataset):
    def __init__(self, samples, targets, transform):
        self.samples = samples
        self.targets = targets
        self.transform = transform

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        path = self.samples[idx]
        label = self.targets[idx]
        image = Image.open(path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        label = torch.tensor(label)
        return image, label


In [10]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.autograd import Function
import numpy as np

class GradReverse(Function):
    @staticmethod
    def forward(ctx, x, lambda_):
        ctx.lambda_ = lambda_
        return x.view_as(x)
    @staticmethod
    def backward(ctx, grad_output):
        return grad_output.neg() * ctx.lambda_, None

def grad_reverse(x, lambda_):
    return GradReverse.apply(x, lambda_)

class DomainDiscriminator(nn.Module):
    def __init__(self, input_dim, hidden_dim=256):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Linear(hidden_dim, hidden_dim//2),
            nn.ReLU(),
            nn.Linear(hidden_dim//2, 1),
            # nn.Sigmoid()
        )
    def forward(self, x):
        return self.net(x)

class GatedAdapter(nn.Module):
    def __init__(self, dim):
        super().__init__()
        hidden = dim * 2
        self.adapter = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, dim)
        )
        # self.gate = nn.Parameter(torch.tensor(1.0))  # learnable weight

    def forward(self, x):
        # return x + self.gate * self.adapter(x)
        return x + self.adapter(x)


class CLIP_DA(nn.Module): ## LD: Logits Deconfusion

    def __init__(self, embed_dim, clip_model):
        super().__init__()
     
        feature_dim = embed_dim  # 1024 / 512
        self.clip_model = clip_model
  
        self.image_adapter = GatedAdapter(feature_dim)
        self.text_adapter = GatedAdapter(feature_dim)
        # self.image_adapter = PromptAwareMoEAdapter(feature_dim)
        # self.text_adapter = PromptAwareMoEAdapter(feature_dim)

        # self.logit_scale = nn.Parameter(torch.ones([]) * np.log(1 / 0.07))
        self.logit_scale = nn.Parameter(torch.tensor(np.log(clip_model.logit_scale.exp().cpu().detach().numpy()))) 
        # self.logit_scale = nn.Parameter(torch.tensor(np.log(50)))
        # self.logit_scale.requires_grad = False


        self.initialize_parameters()

    def initialize_parameters(self):

        for param in self.clip_model.parameters():
            param.requires_grad = False

    def forward_OriCLIP(self, image, text):
        image_features = self.encode_image(image)
        text_features = self.encode_text(text)

        # normalized features
        image_features = image_features / image_features.norm(dim=-1, keepdim=True)
        text_features = text_features / text_features.norm(dim=-1, keepdim=True)

        # cosine similarity as logits
        logit_scale = self.logit_scale.exp()
        logits_per_image = logit_scale * image_features @ text_features.t()
        logits_per_text = logits_per_image.t()

        return logits_per_image, logits_per_text
    
    def encode_text(self,text_tokens):
        text_feats = self.clip_model.encode_text(text_tokens)
        # text_feats = text_feats + self.adapter["adapter_2"](text_feats) ## If adapt text features
        return text_feats

    def forward(self, images, text_feats):  ## Few-shot ensemble text prompts
  
        image_feats = self.clip_model.encode_image(images)
        image_feats = self.image_adapter(image_feats)
        image_logits = F.normalize(image_feats, dim=-1)

        # text_feats = self.clip_model.encode_text(text) ## cause we need to ensemble prompts before
        text_feats =  self.text_adapter(text_feats) ## adapt text features option
        text_logits = F.normalize(text_feats, dim=-1)

        return image_logits,text_logits

def contrastive_loss(img_feats, txt_feats, logit_scale):
    logits_per_image = logit_scale.exp() * (img_feats @ txt_feats.T)  # [B, dim]
    logits_per_text = logits_per_image.T  # [B, dim]

    labels = torch.arange(img_feats.size(0)).to(img_feats.device)  

    loss_i = F.cross_entropy(logits_per_image, labels)
    loss_t = F.cross_entropy(logits_per_text, labels)
    return (loss_i + loss_t) / 2

## AAM

In [ ]:
## 2026
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from collections import defaultdict
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import confusion_matrix, roc_auc_score


## 2026 Clean AAM: Adaptive Multi-level Alignment Model

import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
from collections import defaultdict
from sklearn.metrics import roc_auc_score, confusion_matrix


# =========================================================
# 1. Feature Extraction
# =========================================================
def extract_cls_features(model, images, layers):
    """
    Extract CLS-token features from selected visual transformer layers.
    Features are extracted before projection.
    Returns: list of tensors, each [B, C]
    """
    features = {}
    hooks = []

    def get_hook(idx):
        def hook(module, input, output):
            features[idx] = output[:, 0]
        return hook

    trunk = model.visual.trunk

    for idx in layers:
        hooks.append(trunk.blocks[idx].register_forward_hook(get_hook(idx)))

    _ = model.encode_image(images)

    for h in hooks:
        h.remove()

    return [features[idx] for idx in layers]


def extract_projected_cls_features(model, images, layers):
    """
    Extract projected CLS-token features from selected layers.
    Returns: list of tensors, each [B, D]
    """
    raw_features = {}
    hooks = []

    def get_hook(idx):
        def hook(module, input, output):
            raw_features[idx] = output[:, 0]
        return hook

    trunk = model.visual.trunk

    for idx in layers:
        hooks.append(trunk.blocks[idx].register_forward_hook(get_hook(idx)))

    _ = model.encode_image(images)

    for h in hooks:
        h.remove()

    projected_features = []

    for idx in layers:
        feat = model.visual.head(raw_features[idx])
        feat = F.normalize(feat, dim=-1)
        projected_features.append(feat)

    return projected_features


# =========================================================
# 2. Dynamic Layer Selection
# =========================================================
def compute_layer_scores(model, train_loader, candidate_layers, device):
    """
    Score candidate layers by inter-class separability.
    Higher score means better class separation.
    """
    layer_scores = {}

    with torch.no_grad():
        feats_per_layer = {l: defaultdict(list) for l in candidate_layers}

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            layer_feats = extract_cls_features(model, images, candidate_layers)

            for i, layer in enumerate(candidate_layers):
                feats = F.normalize(layer_feats[i], dim=-1)

                for j, y in enumerate(labels):
                    feats_per_layer[layer][y.item()].append(feats[j])

        for layer in candidate_layers:
            class_feats = feats_per_layer[layer]
            class_ids = sorted(class_feats.keys())

            inter_sims = []

            for i in range(len(class_ids)):
                for j in range(i + 1, len(class_ids)):
                    proto_i = torch.stack(class_feats[class_ids[i]]).mean(dim=0)
                    proto_j = torch.stack(class_feats[class_ids[j]]).mean(dim=0)

                    sim = F.cosine_similarity(proto_i, proto_j, dim=0).item()
                    inter_sims.append(sim)

            if len(inter_sims) > 0:
                layer_scores[layer] = 1.0 - np.mean(inter_sims)
            else:
                layer_scores[layer] = 0.0

    return layer_scores


def select_best_layers(layer_scores, top_k=4, final_layer=11):
    """
    Select top-K layers according to support-set scores.
    The final layer can be forced to preserve high-level semantics.
    """
    items = list(layer_scores.items())

    if final_layer is not None and final_layer in layer_scores:
        items_wo_final = [(l, s) for l, s in items if l != final_layer]
        items_wo_final = sorted(items_wo_final, key=lambda x: x[1], reverse=True)

        selected = [l for l, _ in items_wo_final[:max(top_k - 1, 0)]]
        selected.append(final_layer)
    else:
        items = sorted(items, key=lambda x: x[1], reverse=True)
        selected = [l for l, _ in items[:top_k]]

    selected = sorted(list(set(selected)))
    return selected


def compute_layer_weights(layer_scores, selected_layers, tau=1.0, mode="softmax"):
    """
    Compute adaptive layer weights.
    """
    scores = np.array(
        [layer_scores.get(l, 1e-6) for l in selected_layers],
        dtype=np.float64
    )

    if mode == "softmax":
        scores = scores / max(tau, 1e-6)
        scores = scores - scores.max()
        weights = np.exp(scores)
        weights = weights / weights.sum()

    elif mode == "linear":
        scores = np.maximum(scores, 1e-12)
        weights = scores / scores.sum()

    else:
        raise ValueError(f"Unsupported weight mode: {mode}")

    return weights.tolist()


def prepare_aam_layer_config(
    model,
    train_loader,
    device,
    candidate_layers=[5, 6, 7, 8, 9, 10, 11],
    top_k=4,
    final_layer=11,
    tau=1.0,
    weight_mode="softmax",
):
    """
    Prepare dynamic layer selection and weighting for AAM.
    """
    layer_scores = compute_layer_scores(
        model=model,
        train_loader=train_loader,
        candidate_layers=candidate_layers,
        device=device
    )

    selected_layers = select_best_layers(
        layer_scores=layer_scores,
        top_k=top_k,
        final_layer=final_layer
    )

    layer_weights = compute_layer_weights(
        layer_scores=layer_scores,
        selected_layers=selected_layers,
        tau=tau,
        mode=weight_mode
    )

    layer_config = {
        "layer_scores": layer_scores,
        "selected_layers": selected_layers,
        "layer_weights": layer_weights,
    }

    return layer_config


# =========================================================
# 3. Model
# =========================================================
class ResidualAdapter(nn.Module):
    def __init__(self, dim):
        super().__init__()

        hidden = dim * 2

        self.mlp = nn.Sequential(
            nn.Linear(dim, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, hidden),
            nn.ReLU(inplace=True),
            nn.Linear(hidden, dim)
        )

    def forward(self, x):
        return x + self.mlp(x)


class AAM(nn.Module):
    """
    Adaptive Multi-level Alignment Model.

    Frozen BioMedCLIP encoders.
    Trainable:
    - visual multi-level adapter
    - text adapter
    - logit scale
    """
    def __init__(
        self,
        clip_model,
        embed_dim=512,
        selected_layers=[5, 7, 9, 11],
    ):
        super().__init__()

        self.clip_model = clip_model
        self.embed_dim = embed_dim
        self.selected_layers = selected_layers

        self.visual_adapter = ResidualAdapter(embed_dim)
        self.text_adapter = ResidualAdapter(embed_dim)

        self.logit_scale = nn.Parameter(
            torch.tensor(
                np.log(clip_model.logit_scale.exp().detach().cpu().item()),
                dtype=torch.float32
            )
        )

        self.freeze_clip()

    def freeze_clip(self):
        for p in self.clip_model.parameters():
            p.requires_grad = False

    def adapt_text_features(self, text_feats):
        text_feats = self.text_adapter(text_feats)
        text_feats = F.normalize(text_feats, dim=-1)
        return text_feats

    def forward_multilevel(self, images):
        feats = extract_projected_cls_features(
            self.clip_model,
            images,
            self.selected_layers
        )

        feats = torch.stack(feats, dim=1)          # [B, L, D]
        feats = self.visual_adapter(feats)         # [B, L, D]
        feats = F.normalize(feats, dim=-1)

        return feats


# =========================================================
# 4. Text Features and Logits
# =========================================================
@torch.no_grad()
def build_class_text_features(
    clip_model,
    tokenizer,
    class_names,
    prompt_templates,
    device
):
    prompts = [
        template.format(class_name)
        for class_name in class_names
        for template in prompt_templates
    ]

    tokens = tokenizer(prompts).to(device)

    text_feats = clip_model.encode_text(tokens)
    text_feats = text_feats.view(len(class_names), len(prompt_templates), -1)
    text_feats = text_feats.mean(dim=1)
    text_feats = F.normalize(text_feats, dim=-1)

    return text_feats


def aggregate_layer_logits(raw_logits, layer_weights=None, pooling="weighted"):
    """
    raw_logits: [B, L, C]
    """
    B, L, C = raw_logits.shape

    if pooling == "mean":
        return raw_logits.mean(dim=1)

    elif pooling == "max":
        return raw_logits.max(dim=1).values

    elif pooling == "weighted":
        if layer_weights is None:
            weights = torch.ones(
                L,
                device=raw_logits.device,
                dtype=raw_logits.dtype
            ) / L
        else:
            weights = torch.tensor(
                layer_weights,
                device=raw_logits.device,
                dtype=raw_logits.dtype
            )
            weights = weights / weights.sum()

        return (raw_logits * weights.view(1, L, 1)).sum(dim=1)

    else:
        raise ValueError(f"Unsupported pooling: {pooling}")


def aam_logits(
    model,
    images,
    class_text_feats,
    layer_weights=None,
    pooling="weighted"
):
    """
    Compute AAM multi-level image-text logits.
    """
    visual_feats = model.forward_multilevel(images)      # [B, L, D]
    text_feats = model.adapt_text_features(class_text_feats)

    raw_logits = model.logit_scale.exp() * torch.einsum(
        "bld,cd->blc",
        visual_feats,
        text_feats
    )

    logits = aggregate_layer_logits(
        raw_logits,
        layer_weights=layer_weights,
        pooling=pooling
    )

    return logits, raw_logits


# =========================================================
# 5. Training
# =========================================================
def train_aam(
    clip_model,
    tokenizer,
    train_loader,
    class_names,
    prompt_templates,
    device,
    embed_dim=512,
    candidate_layers=[5, 6, 7, 8, 9, 10, 11],
    top_k=4,
    final_layer=11,
    tau_layer=1.0,
    weight_mode="softmax",
    lr=1e-3,
    weight_decay=1e-4,
    num_epochs=30,
    pooling="weighted",
):
    """
    Train AAM on the support set.
    """

    layer_config = prepare_aam_layer_config(
        model=clip_model,
        train_loader=train_loader,
        device=device,
        candidate_layers=candidate_layers,
        top_k=top_k,
        final_layer=final_layer,
        tau=tau_layer,
        weight_mode=weight_mode
    )

    selected_layers = layer_config["selected_layers"]
    layer_weights = layer_config["layer_weights"]

    print("Selected layers:", selected_layers)
    print("Layer weights:", [round(w, 4) for w in layer_weights])

    model = AAM(
        clip_model=clip_model,
        embed_dim=embed_dim,
        selected_layers=selected_layers
    ).to(device)

    optimizer = torch.optim.AdamW(
        model.parameters(),
        lr=lr,
        weight_decay=weight_decay,
        eps=1e-4
    )

    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
        optimizer,
        T_max=num_epochs * len(train_loader)
    )

    with torch.no_grad():
        class_text_feats = build_class_text_features(
            clip_model=clip_model,
            tokenizer=tokenizer,
            class_names=class_names,
            prompt_templates=prompt_templates,
            device=device
        )

    best_state = None
    best_loss = float("inf")

    for epoch in range(num_epochs):
        model.train()

        running_loss = 0.0
        total_samples = 0

        for images, labels in train_loader:
            images = images.to(device)
            labels = labels.to(device)

            logits, _ = aam_logits(
                model=model,
                images=images,
                class_text_feats=class_text_feats,
                layer_weights=layer_weights,
                pooling=pooling
            )

            loss = F.cross_entropy(logits, labels)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            scheduler.step()

            batch_size = images.size(0)
            running_loss += loss.item() * batch_size
            total_samples += batch_size

        avg_loss = running_loss / max(total_samples, 1)

        if avg_loss < best_loss:
            best_loss = avg_loss
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }

        print(
            f"Epoch {epoch+1:02d} | "
            f"loss: {avg_loss:.4f} | "
            f"logit_scale: {model.logit_scale.exp().item():.4f}"
        )

    if best_state is not None:
        model.load_state_dict(best_state)

    train_meta = {
        "prompt_templates": prompt_templates,
        "layer_config": layer_config,
    }

    return model, train_meta


# =========================================================
# 6. Evaluation
# =========================================================
@torch.no_grad()
def evaluate_aam(
    model,
    clip_model,
    data_loader,
    tokenizer,
    class_names,
    prompt_templates,
    device,
    layer_weights=None,
    pooling="weighted",
    return_details=False
):
    model.eval()

    class_text_feats = build_class_text_features(
        clip_model=clip_model,
        tokenizer=tokenizer,
        class_names=class_names,
        prompt_templates=prompt_templates,
        device=device
    )

    all_labels = []
    all_logits = []
    all_probs = []
    all_preds = []

    total_loss = 0.0
    total_samples = 0

    for images, labels in data_loader:
        images = images.to(device)
        labels = labels.to(device)

        logits, raw_logits = aam_logits(
            model=model,
            images=images,
            class_text_feats=class_text_feats,
            layer_weights=layer_weights,
            pooling=pooling
        )

        probs = F.softmax(logits, dim=1)
        preds = probs.argmax(dim=1)

        loss = F.cross_entropy(logits, labels)

        batch_size = images.size(0)
        total_loss += loss.item() * batch_size
        total_samples += batch_size

        all_labels.append(labels.cpu())
        all_logits.append(logits.cpu())
        all_probs.append(probs.cpu())
        all_preds.append(preds.cpu())

    all_labels = torch.cat(all_labels).numpy()
    all_logits = torch.cat(all_logits).numpy()
    all_probs = torch.cat(all_probs).numpy()
    all_preds = torch.cat(all_preds).numpy()

    results = {
        "loss": total_loss / max(total_samples, 1),
        "acc": (all_preds == all_labels).mean(),
    }

    try:
        labels_onehot = F.one_hot(
            torch.tensor(all_labels),
            num_classes=len(class_names)
        ).numpy()

        auc = roc_auc_score(
            labels_onehot,
            all_probs,
            average="macro",
            multi_class="ovr"
        )

        results["auc"] = auc

    except Exception:
        results["auc"] = np.nan

    if return_details:
        results.update({
            "labels": all_labels,
            "logits": all_logits,
            "probs": all_probs,
            "preds": all_preds,
        })

    return results


# =========================================================
# 9. Plotting
# =========================================================
def plot_eval_results(labels, probs, preds, class_names):
    acc = (preds == labels).mean()
    print(f"Accuracy: {acc:.4f}")

    cm = confusion_matrix(labels, preds)

    plt.figure(figsize=(10, 8))
    sns.heatmap(
        cm,
        annot=True,
        fmt='d',
        cmap='Blues',
        xticklabels=class_names,
        yticklabels=class_names
    )
    plt.xlabel("Predicted Label")
    plt.ylabel("True Label")
    plt.xticks(rotation=45, ha="right")
    plt.yticks(rotation=0)
    plt.tight_layout()
    plt.show()

    num_classes = len(class_names)
    labels_onehot = F.one_hot(torch.tensor(labels), num_classes=num_classes).numpy()

    try:
        aucs = roc_auc_score(labels_onehot, probs, average=None, multi_class="ovr")
        for i, auc in enumerate(aucs):
            print(f"Class {class_names[i]}: AUROC = {auc:.4f}")
        print(f"Averaged AUROC: {aucs.mean():.4f}")
    except Exception as e:
        print("Error when calculating AUROC:", e)

In [62]:
# prompt_templates = [
#         "a photo of chest X-ray showing {}",
#         "a chest X-ray image showing {}.",
#         # "evidence of {} in lungs",
#         # "radiographic signs of {}.",
#         # "a patient diagnosed with {} in chest X-ray."
#     ]
prompt_templates = [
        "a photo of Ultrasound showing {}",
        "an Ultrasound image showing {}.",
    ]
# prompt_templates = [
#         "a photo of CT showing {}",
#         "a CT image showing {}.",
#     ]
# prompt_templates = [
#         "a photo of histopathology showing {}",
#         "a histopathology image showing {}.",
#     ]

In [63]:
# template = 'this is a photo of chest X-ray showing'
# text_tokens = tokenizer([template + l for l in class_names]).to(device)
from dataset.dataset import Chestxray14_Dataset
import torch
import torch.nn.functional as F
from torchvision.datasets import ImageFolder
from torchvision import transforms
from torch.utils.data import DataLoader
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix, roc_auc_score, ConfusionMatrixDisplay
import matplotlib.pyplot as plt
from tqdm import tqdm

device = "cuda" 
# Basic transform compatible with CLIP input size & normalization
from torchvision import transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.RandomHorizontalFlip(p=0.5),           
    transforms.Lambda(lambda img: img.convert("RGB")),      
    transforms.ColorJitter(brightness=0.2, contrast=0.2), 
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.48145466, 0.4578275, 0.40821073),
                         std=(0.26862954, 0.26130258, 0.27577711))  
])
test_transform = transforms.Compose([
    transforms.Resize((224, 224)),       
    transforms.Lambda(lambda img: img.convert("RGB")),      
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.48145466, 0.4578275, 0.40821073),
                         std=(0.26862954, 0.26130258, 0.27577711))  
])

# dataset_root = r"D:\MedicalVLMs\data\COVIDxCXR4"
# dataset_root = r'D:\MedicalVLMs\data\Chest X-Ray Images (Pneumonia)'
# dataset_root = r'D:\MedicalVLMs\data\CheXpert_5x1000'
# dataset_root = r"D:\MedicalVLMs\data\Tuberculosis Chest X-rays (Shenzhen)"
dataset_root = r"D:\MedicalVLMs\exp_datasets\ultrasound\BUSI"
# dataset_root = r'D:\MedicalVLMs\exp_datasets\BTMRI'
# dataset_root = r'D:\MedicalVLMs\exp_datasets\CTKi'
# dataset_root = r'D:\MedicalVLMs\exp_datasets\SCCT'
# dataset_root = r'D:\MedicalVLMs\exp_datasets\LC25000'
fewshot_dataset = FewShotImageFolder(root=dataset_root, num_shots=16, transform=transform, test_transform = test_transform, seed=2024)

from torch.utils.data import DataLoader
train_loader = DataLoader(fewshot_dataset, batch_size=32, shuffle=True)

test_dataset = fewshot_dataset.get_test_set()
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

print(f"Train size: {len(fewshot_dataset)}")
print(f"Test size: {len(test_dataset)}")


idx_to_folder = {v: k for k, v in fewshot_dataset.class_to_idx.items()}
class_names = [folder_name for idx, folder_name in sorted(idx_to_folder.items())]
print(class_names)

Train size: 48
Test size: 732
['benign tumor', 'malignant tumor', 'normal']


In [ ]:
import numpy as np
import torch
from sklearn.metrics import roc_auc_score

shot_list = [1, 2, 4, 8, 16]
seed_list = list(range(2000, 2050))

results_all = {
    "shot": [],
    "aam_acc_mean": [],
    "aam_acc_std": [],
    "aam_auc_mean": [],
    "aam_auc_std": [],
}

for shot in shot_list:
    print("\n" + "=" * 60)
    print(f"Running {shot}-shot AAM experiment (multi-seed)")
    print("=" * 60)

    aam_acc_list = []
    aam_auc_list = []

    for seed in seed_list:
        print(f"\n--- Seed {seed} ---")

        # =========================
        # 1. Dataset
        # =========================
        fewshot_dataset = FewShotImageFolder(
            root=dataset_root,
            num_shots=shot,
            transform=transform,
            test_transform=test_transform,
            seed=seed
        )

        train_loader = DataLoader(
            fewshot_dataset,
            batch_size=32,
            shuffle=True
        )

        test_dataset = fewshot_dataset.get_test_set()
        test_loader = DataLoader(
            test_dataset,
            batch_size=32,
            shuffle=False
        )

        idx_to_folder = {v: k for k, v in fewshot_dataset.class_to_idx.items()}
        class_names = [
            folder_name
            for idx, folder_name in sorted(idx_to_folder.items())
        ]

        # =========================
        # 2. Train AAM
        # =========================
        aam_model, train_meta = train_aam(
            clip_model=model,
            tokenizer=tokenizer,
            train_loader=train_loader,
            class_names=class_names,
            prompt_templates=prompt_templates,
            device=device,
            embed_dim=512,
            candidate_layers=[5, 6, 7, 8, 9, 10, 11],
            top_k=4,
            final_layer=11,
            tau_layer=1.0,
            weight_mode="softmax",
            lr=1e-3,
            weight_decay=1e-4,
            num_epochs=50,
            pooling="weighted",
        )

        aam_model.to(device)

        layer_config = train_meta["layer_config"]

        # =========================
        # 3. Evaluate AAM
        # =========================
        test_metrics = evaluate_aam(
            model=aam_model,
            clip_model=model,
            data_loader=test_loader,
            tokenizer=tokenizer,
            class_names=class_names,
            prompt_templates=prompt_templates,
            device=device,
            layer_weights=layer_config["layer_weights"],
            pooling="weighted",
            return_details=True
        )

        aam_acc = test_metrics["acc"]
        aam_auc = test_metrics.get("auc", np.nan)

        aam_acc_list.append(aam_acc)
        aam_auc_list.append(aam_auc)

        print(
            f"Seed {seed} | "
            f"AAM Acc: {aam_acc:.4f} | "
            f"AAM AUC: {aam_auc:.4f}"
        )

    # =========================
    # 4. Aggregate
    # =========================
    def mean_std(x):
        x = np.array(x, dtype=float)
        return np.nanmean(x), np.nanstd(x, ddof=1)

    acc_mean, acc_std = mean_std(aam_acc_list)
    auc_mean, auc_std = mean_std(aam_auc_list)

    results_all["shot"].append(shot)
    results_all["aam_acc_mean"].append(acc_mean)
    results_all["aam_acc_std"].append(acc_std)
    results_all["aam_auc_mean"].append(auc_mean)
    results_all["aam_auc_std"].append(auc_std)

    print("\n================ RESULT ================")
    print(f"{shot}-shot:")
    print(f"AAM Acc: {acc_mean:.4f} ± {acc_std:.4f}")
    print(f"AAM AUC: {auc_mean:.4f} ± {auc_std:.4f}")